[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR_USERNAME/Latent/blob/main/notebooks/colab/Stage_4_Singlish_to_English.ipynb)

---


In [ ]:
# ══════════════════════════════════════════
# Google Colab Setup
# ══════════════════════════════════════════
# Run this cell first to set up the environment in Google Colab.

import os
import sys

# Check if running in Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Mount Google Drive for data persistence
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Set project root on Google Drive
    PROJECT_ROOT = '/content/drive/MyDrive/Latent'
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/intermediate_data', exist_ok=True)
    os.makedirs(f'{PROJECT_ROOT}/utilities', exist_ok=True)
    
    # Clone or update the repo
    if not os.path.exists('/content/Latent'):
        !git clone https://github.com/YOUR_USERNAME/Latent.git /content/Latent
    
    # Install dependencies
    !pip install -q spacy nltk ftfy emoji langdetect sentence-transformers rank-bm25 tqdm
    !python -m spacy download en_core_web_sm
    
    # Set working directory
    os.chdir('/content/Latent')
    print(f'Working directory: {os.getcwd()}')
    print(f'Data directory: {PROJECT_ROOT}')
else:
    PROJECT_ROOT = '.'
    print('Running locally. Ensure dependencies are installed via: pip install -r requirements.txt')


# Stage 4: Singlish to English Conversion

### Project Structure

```bash
Latent/
    │
    ├─ data/
    │   ├─ inverted_index/
    │   │   ├─ inverted_index_tfidf_titles.json
    │   │   └─ inverted_index_tfidf_fulltext.json
    │   │
    │   ├─ models/
    │   │   ├─ sentence_bert_model/
    │   │   ├─ bm25_comments_model.joblib
    │   │   ├─ bm25_fulltext_model.joblib
    │   │   ├─ bm25_titles_model.joblib
    │   │   ├─ tfidf_comments_vectorizer.joblib
    │   │   └─ tfidf_posts_vectorizer.joblib
    │   │
    │   ├─ vector_database/
    │   │   ├─ bert_comments_embeddings.npy
    │   │   ├─ bert_contents_embeddings.npy
    │   │   ├─ bert_fulltext_embeddings.npy
    │   │   ├─ bert_titles_embeddings.npy
    │   │   ├─ bm25_comments.npz
    │   │   ├─ bm25_contents.npz
    │   │   ├─ bm25_fulltext.npz
    │   │   ├─ bm25_titles.npz
    │   │   ├─ tfidf_comments.npz
    │   │   ├─ tfidf_contents.npz
    │   │   ├─ tfidf_fulltext.npz
    │   │   └─ tfidf_titles.npz
    │   │
    │   ├─ PostVault.csv
    │   ├─ CommentVault.csv
    │   └─ raw_data/
    │
    ├─ sentiment_plots/
    │   ├─ emotion_plots/
    │   ├─ emotion_dashboard.py
    │   └─ plot_emotion_summary.py
    │
    ├─ Stage_0_Introduction.ipynb                               
    ├─ Stage_1_Data_Collection_and_Data_Cleaning.ipynb
    ├─ Stage_2_POS_and_NER_Tagging.ipynb
    ├─ Stage_3_Singlish_Normalisation.ipynb
-----------------------------------------------------------------
│   ├─ Stage_4_Singlish_to_English_Conversion.ipynb             │
-----------------------------------------------------------------            
    ├─ Stage_5_Common_Normalisation.ipynb
    ├─ Stage_6_Vector_Space_Model_and_Inverted_Index.ipynb
    ├─ Stage_7_Sentiment_Analysis.ipynb
    ├─ Stage_8_Clustering_and_Visualization.ipynb       
    ├─ Stage_9_Document_Search.ipynb
    └─ utilities/
        │
        ├─ pp_class.py
        ├─ singlish_dictionary.json
        ├─ singlish_regex_to_text.txt
        └─ slang_dictionary.csv
```

## To access the mounted folder directly.

### Step 1: Add the Shared Folder to your Google Drive (if you haven't already)

1.  **Open Google Drive** (drive.google.com) in your web browser.
2.  Go to the **'Shared with me'** section.
3.  Locate the folder that was shared with you.
4.  **Right-click** on the shared folder.
5.  Select **'Add shortcut to Drive'** (or 'Add to My Drive', depending on the interface). Choose a location within 'My Drive' if prompted, or simply add it to the root of 'My Drive'.

This creates a shortcut to the shared folder in your 'My Drive', making it accessible when Colab mounts your personal Drive.
### Step 2: Access the Shared Folder in Colab (No action needed)

Once your Drive is mounted, you can navigate to the shared folder. If you added a shortcut, it will appear in your 'My Drive' just like any other folder. The path will typically be `/content/gdrive/MyDrive/YourSharedFolderName`.

It should be able to run the script below already

In [ ]:
# -----------
# Mount Drive
# -----------
from google.colab import drive
drive.mount('/content/drive')

## Dependencies

### Install Dependencies

In [ ]:
!pip install emoji ftfy langdetect import-ipynb

### Change Directory

In [ ]:
# Path
# ----
FOLDER_PATH: str = "/content/drive/MyDrive/Latent/"

import os
os.chdir(FOLDER_PATH)

### Import Essential Library

In [ ]:
import re
import json
import import_ipynb
import pandas as pd
from tqdm import tqdm
from Stage_2_POS_and_NER_Tagging import pos_tagger
from Stage_0_Introduction import display_first_text_column_head, save_dfs_to_csv, output_base_dir, save_single_df_to_csv

## Singlish Conversion

### Load Data

In [ ]:
# ---------------
# Load Clean Data
# ---------------
df_posts = pd.read_csv('/content/drive/MyDrive/Latent/data/PostVault.csv', keep_default_na=False)
df_comments = pd.read_csv('/content/drive/MyDrive/Latent/data/CommentVault.csv', keep_default_na=False)
# ------------------------
# Load Singlish Dictionary
# ------------------------
dictionary_path = '/content/drive/MyDrive/Latent/utilities/singlish_dictionary.json'
with open(dictionary_path, "r") as f:
    sing_dict = json.load(f)

### Singlish to English Conversion

In [ ]:
def simple_singlish_to_english_conversion(text):
    """
    Converts Singlish phrases to English using a predefined dictionary,
    applying direct replacements without POS tagging.

    Also returns the number of conversions made.

    Args:
        text (str): The input text string which may contain Singlish.

    Returns:
        tuple: A tuple containing:
            - str: The text with Singlish terms converted to English based on the map.
            - int: The total number of conversions made.
    """
    if not isinstance(text, str):
        return "", 0

    converted_text = text
    conversion_count = 0
    for pattern, replacement in sing_dict.items():
        try:

            new_text, count = re.subn(
                f"\\b{pattern}\\b",
                replacement,
                converted_text,
                flags=re.IGNORECASE
            )
            if count > 0:
                converted_text = new_text
                conversion_count += count
        except re.error:
            pass

    return converted_text, conversion_count

def convert_singlish_columns(df, text_columns):

    if 'singlish_count' not in df.columns:
        df['singlish_count'] = 0
    else:
        # Attempt to convert to numeric, coercing errors to NaN, fill NaN with 0, then cast to int.
        df['singlish_count'] = pd.to_numeric(df['singlish_count'], errors='coerce').fillna(0).astype(int)

    for col in text_columns:
        if col in df.columns:
            # Apply conversion
            conversion_results = df[col].astype(str).apply(simple_singlish_to_english_conversion)

            # Extract converted text
            df[f'english_converted_' + col.replace('singlish_normalized_', '')] = conversion_results.apply(lambda x: x[0])

            # Accumulate counts
            df['singlish_count'] += conversion_results.apply(lambda x: x[1])

    return df


In [ ]:
# ---------
# Unit Test
# ---------
print("Demonstrating simple_singlish_to_english_conversion")
print("---------------------------------------------------")
print()

sentence1 = "Wah lau, so sian today, bo liao."
sentence2 = "Can I have a packet of milo peng, please?"
sentence3 = "Dun play play, confirm plus chop very good!"

converted_s1, count_s1 = simple_singlish_to_english_conversion(sentence1)
print(f"Original: {sentence1}")
print(f"Converted: {converted_s1}")
print(f"Conversions: {count_s1}\n")

converted_s2, count_s2 = simple_singlish_to_english_conversion(sentence2)
print(f"Original: {sentence2}")
print(f"Converted: {converted_s2}")
print(f"Conversions: {count_s2}\n")

converted_s3, count_s3 = simple_singlish_to_english_conversion(sentence3)
print(f"Original: {sentence3}")
print(f"Converted: {converted_s3}")
print(f"Conversions: {count_s3}\n")

print("Note")
print("----")
print("This function performs direct replacements using the `sing_dict`.")
print("For a more robust conversion that preserves proper nouns, you should use the 'normalize_singlish_with_pos' function defined previously.")

#### PostVault

In [ ]:
df_posts.head()

In [ ]:
text_columns = ['singlish_normalized_title', 'singlish_normalized_content']
df_posts = convert_singlish_columns(df_posts, text_columns)

#### CommentVault

In [ ]:
text_columns = ['singlish_normalized_text']
df_comments = convert_singlish_columns(df_comments, text_columns)

### Save Files

In [ ]:
print("---------")
print("PostVault")
print("---------")
display(df_posts.head())
save_single_df_to_csv(df_posts, "/content/drive/MyDrive/Latent/data/", "PostVault")

In [ ]:
print("------------")
print("CommentVault")
print("------------")
display(df_comments.head())
save_single_df_to_csv(df_comments, "/content/drive/MyDrive/Latent/data/", "CommentVault")